In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ============================================================
# Parameters
# ============================================================

filename = "A_ir_16k_sp2.h5"

start_time = 6.0   # [ms]
end_time = 10.5    # [ms]

# z slice position (MATLAB indexing starts from 1)
z_pos = 26

# ============================================================
# Load HDF5 data
# ============================================================

with h5py.File(filename, "r") as f:

    # RIR dataset
    rir = f["/rir"][:]

    # Spatial grid coordinates
    grid = f["/grid"][:]

    # Attributes
    fs = float(np.squeeze(f.attrs["fs"]))
    c = float(np.squeeze(f.attrs["c"]))

    # Grid dimensions [Nx, Ny, Nz]
    grid_dim = np.array(
        f.attrs["grid_dimensions"]
    ).astype(int).ravel()

# ============================================================
# Shape correction
# MATLAB -> Python transpose correction
# ============================================================

# Convert grid to shape [N, 3]
if grid.shape[0] == 3 and grid.shape[1] != 3:
    grid = grid.T

num_x, num_y, num_z = grid_dim

num_points = grid.shape[0]

# Convert rir to shape [time, N]
if rir.shape[0] == num_points and rir.shape[1] != num_points:
    rir = rir.T

# ============================================================
# Reshape to 4D array
# [time, x, y, z]
# ============================================================

ir_length = rir.shape[0]

# order="F" reproduces MATLAB reshape ordering
rir4d = rir.reshape(
    (ir_length, num_x, num_y, num_z),
    order="F"
)

# ============================================================
# Normalize amplitude
# ============================================================

rir4d = rir4d / np.max(np.abs(rir4d))

# ============================================================
# Time range for visualization
# ============================================================

t_start = int(start_time / 1000 * fs)
t_end = int(end_time / 1000 * fs)

frames = np.arange(t_start, t_end + 1)

# Python index starts from 0
z_idx = z_pos - 1

# ============================================================
# Create figure
# ============================================================

fig, ax = plt.subplots(figsize=(6, 5))

# Initial frame
p0 = rir4d[frames[0], :, :, z_idx]

# Display image
im = ax.imshow(
    p0.T,
    origin="lower",
    aspect="equal",
    vmin=-0.25,
    vmax=0.25,
    cmap="jet",

    # Physical axis size [m]
    extent=[
        0,
        (num_x - 1) * 0.02,
        0,
        (num_y - 1) * 0.02
    ],
)

fig.colorbar(im, ax=ax)

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")

# ============================================================
# Animation update function
# ============================================================

def update(t_idx):

    # Extract x-y slice at fixed z
    p = rir4d[t_idx, :, :, z_idx]

    # Update image
    im.set_data(p.T)

    # Update title
    ax.set_title(
        f"Time = {t_idx / fs * 1000:.2f} ms"
    )

    return [im]

# ============================================================
# Create animation
# ============================================================

ani = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=100,
    blit=False
)

plt.close(fig)

# ============================================================
# Display animation in Jupyter Notebook
# ============================================================

HTML(ani.to_jshtml())

# Other choice
# plt.show()
# 